# duckdb-kql — run KQL queries on DuckDB

[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/mmaitre314/duckdb-kql?devcontainer_path=.devcontainer%2Fdemo%2Fdevcontainer.json&quickstart=1)

[`duckdb-kql`](https://github.com/mmaitre314/duckdb-kql) translates
[Kusto Query Language](https://learn.microsoft.com/azure/data-explorer/kusto/query/) (KQL)
into [DuckDB](https://duckdb.org/) SQL, so you can run KQL against local files and in-process data with
no cluster and no network.

To get started, either open the GitHub Codespace above or install Python followed by `pip install duckdb-kql[all]`.

In [1]:
import duckdb_kql

print("duckdb-kql", duckdb_kql.__version__)

duckdb-kql 0.0.1.dev2


## Sample data

Connect to an in-memory instance of DuckDB and populate a sample table `Requests`.

In [2]:
import random
from datetime import datetime, timedelta

con = duckdb_kql.connect()

start = datetime(2026, 3, 1)

rows = []
for i in range(2_000):
    service = random.choice(["checkout", "search", "auth", "catalog"])
    status = random.choice([200, 200, 200, 500] if service == "auth" else [200, 200, 200, 200, 503])
    rows.append((
        start + timedelta(seconds=i * 37),
        service,
        random.choice(["us-west-2", "us-east-1", "eu-west-1"]),
        status,
        round(random.lognormvariate(3.2, 0.6), 1),
        f"user-{random.randint(1, 120):03d}",
    ))

con.execute("""
    CREATE TABLE Requests (
        Timestamp TIMESTAMP, Service VARCHAR, Region VARCHAR,
        Status INTEGER, LatencyMs DOUBLE, UserId VARCHAR
    )
""")
con.executemany("INSERT INTO Requests VALUES (?, ?, ?, ?, ?, ?)", rows)

print(duckdb_kql.engine.schema(con))

{'Requests': ['Timestamp', 'Service', 'Region', 'Status', 'LatencyMs', 'UserId']}


## Basic query

Run a simple query.

In [3]:
duckdb_kql.kql(con, """
    Requests
    | where Status >= 500
    | summarize Failures = count(), Users = dcount(UserId) by Service, Region
    | sort by Failures desc
    | take 5
""")

┌──────────┬───────────┬──────────┬───────┐
│ Service  │  Region   │ Failures │ Users │
│ varchar  │  varchar  │  int64   │ int64 │
├──────────┼───────────┼──────────┼───────┤
│ auth     │ us-east-1 │       60 │    47 │
│ auth     │ us-west-2 │       41 │    34 │
│ search   │ us-west-2 │       40 │    34 │
│ checkout │ us-west-2 │       39 │    36 │
│ auth     │ eu-west-1 │       36 │    31 │
└──────────┴───────────┴──────────┴───────┘

## Query with parameters

Run a query with parameters (preventing query injection).

In [5]:
duckdb_kql.kql(con, """
    declare query_parameters(service:string, floor:long);
    
    Requests
    | where Service == service and Status >= floor
    | count
""",
    {"service": "auth", "floor": 500})

┌───────┐
│ Count │
│ int64 │
├───────┤
│   137 │
└───────┘

## KustoClient-compatible query

Run a query using APIs compatible with Kusto client SDK ([azure-kusto-data](https://github.com/Azure/azure-kusto-python)). Useful to unit-test code with KQL queries meant to run against actual Kusto clusters.

In [6]:
from duckdb_kql.kusto import KustoClient
from duckdb_kql.kusto.helpers import dataframe_from_result_table

client = KustoClient(con)

response = client.execute("NetDefaultDB", """
    Requests
    | where Status >= 500
    | summarize Errors = count() by Service
    | sort by Errors desc
""")

table = response.primary_results[0]
print("columns  :", [(c.column_name, c.column_type) for c in table.columns])
print("raw_rows :", table.raw_rows)
print()
dataframe_from_result_table(table)

columns  : [('Service', 'string'), ('Errors', 'long')]
raw_rows : [['auth', 137], ['search', 100], ['catalog', 98], ['checkout', 96]]



,Service,Errors
0,auth,137
1,search,100
2,catalog,98
3,checkout,96


## KQL query translation

Pre-compute the SQL-equivalent query. This removes the runtime dependency on `duckdb_kql`.

In [7]:
print(duckdb_kql.to_sql("""
    Requests
    | where Status >= 500 and Region has "west"
    | summarize Failures = count(), Users = dcount(UserId) by Service
    | sort by Failures desc
    | take 3
"""))

WITH _s0 AS (SELECT * FROM "Requests"),
     _s1 AS (SELECT * FROM _s0 WHERE (("Status" >= CAST(500 AS BIGINT)) AND coalesce(regexp_matches("Region", '(?i)\b' || regexp_escape('west') || '\b'), FALSE))),
     _s2 AS (SELECT "Service" AS "Service", count(*) AS "Failures", count(DISTINCT "UserId") AS "Users" FROM _s1 GROUP BY "Service"),
     _s3 AS (SELECT * FROM _s2 ORDER BY "Failures" DESC NULLS LAST),
     _s4 AS (SELECT * FROM _s3 LIMIT 3)
SELECT * FROM _s4


Validate the syntax of queries.

In [7]:
for diagnostic in duckdb_kql.validate("Requests | wehr Status == 500"):
    print(diagnostic)

1:9: mismatched input '|' expecting {<EOF>, ';'}


Translate KQL queries using the CLI, to run query translation as part of build pipelines for instance.

In [8]:
!duckdb-kql translate demo.kql

-- Generated by duckdb-kql from demo.kql. Do not edit.
--
-- Run with TimeZone set to UTC:  SET TimeZone='UTC';
-- KQL datetimes are UTC, and DuckDB reads the session zone when casting
-- text without an offset. Without it, datetimes are silently shifted
-- rather than rejected.

WITH _s0 AS (SELECT * FROM "Requests"),
     _s1 AS (SELECT * FROM _s0 WHERE ("Status" >= CAST(500 AS BIGINT))),
     _s2 AS (SELECT "Service" AS "Service", count(*) AS "Errors" FROM _s1 GROUP BY "Service")
SELECT * FROM _s2


## Next steps

Documentation:
- **[Getting started](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/getting-started.md)**
- **[API reference](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/api.md)**
- **[KQL support matrix](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/kql-support.md)**
- **[Kusto SDK compatibility](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/kusto-client.md)**
- **[Build-time translation](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/cli.md)**.

Found a query that returns something different from Kusto?
[Open an issue](https://github.com/mmaitre314/duckdb-kql/issues).